In [1]:
import os
import torch
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from mpstemmer import MPStemmer

import gensim
from gensim import corpora
from gensim.utils import simple_preprocess
from pprint import pprint

from transformers import BertTokenizer
from nltk.tokenize import RegexpTokenizer

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
print("Total GPU:", torch.cuda.device_count())
print("Current GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

Total GPU: 1
Current GPU: NVIDIA RTX A5000


In [2]:
path = "../bert_data/id-p2"

train_files = []
val_files = []
test_files = []
for file in os.listdir(path):
    if "bert.pt" in file and "train" in file:
        train_files.append(path + "/" + file)
    elif "bert.pt" in file and "valid" in file:
        val_files.append(path + "/" + file)
    elif "bert.pt" in file and "test" in file:
        test_files.append(path + "/" + file)

train_files = sorted(train_files)
val_files = sorted(val_files)
test_files = sorted(test_files)

In [3]:
train_files = train_files[:]
val_files = val_files[:]
test_files = test_files[:]

In [4]:
train_docs = []

i = 0
for file in train_files:
    print(f"Loading data train {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        train_docs.append(src)
    i = i + 1

Loading data train 0...
Loading data train 1...
Loading data train 2...
Loading data train 3...
Loading data train 4...
Loading data train 5...
Loading data train 6...
Loading data train 7...
Loading data train 8...
Loading data train 9...
Loading data train 10...
Loading data train 11...
Loading data train 12...
Loading data train 13...
Loading data train 14...
Loading data train 15...
Loading data train 16...
Loading data train 17...
Loading data train 18...
Loading data train 19...


In [5]:
val_docs = []

i = 0
for file in val_files:
    print(f"Loading data val {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        val_docs.append(src)
    i = i + 1

Loading data val 0...
Loading data val 1...
Loading data val 2...


In [6]:
test_docs = []

i = 0
for file in test_files:
    print(f"Loading data test {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        test_docs.append(src)
    i = i + 1

Loading data test 0...
Loading data test 1...
Loading data test 2...


In [7]:
# Remove stop words
def remove_stop_words(doc):
    factory = StopWordRemoverFactory()
    stopword = factory.create_stop_word_remover()
    res = stopword.remove(doc)
    return res

In [8]:
rm_train_docs = []
rm_val_docs = []
rm_test_docs = []

# Preprocess only removing stop words
# If we also use stemming, the topic will be non-sense
for doc in train_docs:
    # print("Preprocess train docs...")
    doc = remove_stop_words(doc)
    rm_train_docs.append(doc)

for doc in val_docs:
    # print("Preprocess val docs...")
    doc = remove_stop_words(doc)
    rm_val_docs.append(doc)

for doc in test_docs:
    # print("Preprocess test docs...")
    doc = remove_stop_words(doc)
    rm_test_docs.append(doc)

In [9]:
def tokenize(docs):
    # Split the documents into tokens.
    tokenizer = RegexpTokenizer(r'\w+')
    new_docs = docs.copy()
    for idx in range(len(docs)):
        new_docs[idx] = docs[idx].lower()  # Convert to lowercase.
        new_docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words.
        
    return new_docs

In [10]:
proc_train_docs = tokenize(rm_train_docs)
proc_val_docs = tokenize(rm_train_docs)
proc_test_docs = tokenize(rm_train_docs)

In [11]:
# Create a dictionary representation of the documents.
dictionary = corpora.Dictionary(proc_train_docs)

In [12]:
# Bag-of-words representation of the documents.
corpus = [dictionary.doc2bow(doc) for doc in proc_train_docs]
val_bow = [dictionary.doc2bow(doc) for doc in proc_val_docs]
test_bow = [dictionary.doc2bow(doc) for doc in proc_test_docs]

In [13]:
print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(corpus))

Number of unique tokens: 182143
Number of documents: 38207


In [14]:
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                   level=logging.DEBUG,
                   filename='lda_model.log')

In [15]:
# Train LDA model.
from gensim.models import LdaModel

# Set training parameters.
num_topics = 200
chunksize = 5000
passes = 1
iterations = 400
eval_every = None  # Don't evaluate model perplexity, takes too much time.

# Make an index to word dictionary.
temp = dictionary[0]  # This is only to "load" the dictionary.
id2word = dictionary.id2token

model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    chunksize=chunksize,
    alpha='auto',
    eta='auto',
    iterations=iterations,
    num_topics=num_topics,
    passes=passes,
    eval_every=eval_every
)

In [43]:
train_topic_distr = []
val_topic_distr = []
test_topic_distr = []

# Approximate topic distribution for each doc    
for bow in corpus:
    distr = model.get_document_topics(bow, minimum_probability=0)
    distr.sort(key=lambda x: x[1], reverse=True)
    train_topic_distr.append(distr[:10])

In [57]:
for bow in val_bow:
    distr = model.get_document_topics(bow, minimum_probability=0)
    distr.sort(key=lambda x: x[1], reverse=True)
    val_topic_distr.append(distr[:10])

In [ ]:
for bow in test_bow:
    distr = model.get_document_topics(bow, minimum_probability=0)
    distr.sort(key=lambda x: x[1], reverse=True)
    test_topic_distr.append(distr[:10])

In [16]:
model.print_topics()

[(32,
  '0.045*"max" + 0.037*"boeing" + 0.022*"737" + 0.017*"harun" + 0.016*"am" + 0.011*"khalid" + 0.010*"dibawakan" + 0.010*"faa" + 0.009*"inn" + 0.009*"ana"'),
 (99,
  '0.074*"singa" + 0.036*"mutasi" + 0.031*"varian" + 0.017*"trofi" + 0.010*"pramusim" + 0.010*"arina" + 0.009*"lgbti" + 0.009*"cederanya" + 0.008*"silat" + 0.008*"leverkusen"'),
 (2,
  '0.010*"rikwanto" + 0.007*"daesh" + 0.007*"mirwais" + 0.006*"ypg" + 0.006*"mca" + 0.006*"orang" + 0.006*"kata" + 0.005*"manbij" + 0.005*"tahun" + 0.005*"meditasi"'),
 (122,
  '0.078*"pengungsi" + 0.057*"rohingya" + 0.046*"myanmar" + 0.024*"bangladesh" + 0.023*"kamp" + 0.013*"pbb" + 0.011*"warga" + 0.010*"rakhine" + 0.009*"laba" + 0.008*"perbatasan"'),
 (34,
  '0.081*"harimau" + 0.029*"macan" + 0.018*"bangkai" + 0.015*"jawa" + 0.014*"holmes" + 0.014*"lansia" + 0.014*"tank" + 0.012*"tiananmen" + 0.012*"tutul" + 0.010*"88"'),
 (56,
  '0.087*"budi" + 0.035*"gunawan" + 0.020*"susi" + 0.019*"kapolri" + 0.016*"haiti" + 0.016*"johan" + 0.015*"kac

In [17]:
model.num_topics

200

In [18]:
model.save('best_lda.model')

In [36]:
# Get the topic-word distribution
topic_word_dist = model.get_topics()

topics_raw = {}

# Display the probabilities of each word for each topic
for topic_id, topic in enumerate(topic_word_dist):
    word_probs = [(dictionary[word_id], prob) for word_id, prob in enumerate(topic)]
    word_probs = sorted(word_probs, key=lambda x: x[1], reverse=True)  # Sort by probability
    topics_raw[topic_id] = word_probs[:20]
    print(f"Topic {topic_id}...")

Topic 0...
Topic 1...
Topic 2...
Topic 3...
Topic 4...
Topic 5...
Topic 6...
Topic 7...
Topic 8...
Topic 9...
Topic 10...
Topic 11...
Topic 12...
Topic 13...
Topic 14...
Topic 15...
Topic 16...
Topic 17...
Topic 18...
Topic 19...
Topic 20...
Topic 21...
Topic 22...
Topic 23...
Topic 24...
Topic 25...
Topic 26...
Topic 27...
Topic 28...
Topic 29...
Topic 30...
Topic 31...
Topic 32...
Topic 33...
Topic 34...
Topic 35...
Topic 36...
Topic 37...
Topic 38...
Topic 39...
Topic 40...
Topic 41...
Topic 42...
Topic 43...
Topic 44...
Topic 45...
Topic 46...
Topic 47...
Topic 48...
Topic 49...
Topic 50...
Topic 51...
Topic 52...
Topic 53...
Topic 54...
Topic 55...
Topic 56...
Topic 57...
Topic 58...
Topic 59...
Topic 60...
Topic 61...
Topic 62...
Topic 63...
Topic 64...
Topic 65...
Topic 66...
Topic 67...
Topic 68...
Topic 69...
Topic 70...
Topic 71...
Topic 72...
Topic 73...
Topic 74...
Topic 75...
Topic 76...
Topic 77...
Topic 78...
Topic 79...
Topic 80...
Topic 81...
Topic 82...
Topic 83...
To

In [37]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p2")
vocab = tokenizer.get_vocab()

In [38]:
# Convert words into tokens
# Ex: {0: [('korea', 0.1), ('utara', 0.2)]}
# --> {0: [('ko', 0.1), ('##rea', 0.1), ('utara', 0.2)]}
topics_tokenized = {}
for topic, words in topics_raw.items():
    if topic < 0:
        continue
    topics_tokenized[topic] = []    
    for word, score in words:
        tokens = tokenizer.tokenize(word)
        for tok in tokens:
            topics_tokenized[topic].append((tok, score))

In [39]:
# Clean topics that have same tokens inside it and assign the highest score
# Ex: {0: [('ko', 0.1), ('##rea', 0.1), ('utara', 0.2), ('ko', 0.01)]}
# --> {0: [('ko', 0.1), ('##rea', 0.1), ('utara', 0.2)]}
topics_tokenized_clean = {}
for topic, tokens in topics_tokenized.items():
    seen_tok = []
    topics_tokenized_clean[topic] = []
    for tok, score in tokens:
        if tok in seen_tok:
            continue
        else:
            seen_tok.append(tok)
            topics_tokenized_clean[topic].append((tok, score))

In [40]:
MAX_TOKENS = 50

topics_T = []
for topic in topics_tokenized_clean:
    li = []
    for token_tuple in topics_tokenized_clean[topic]:
        token, score = token_tuple
        li.append((token, score))

    if len(li) < MAX_TOKENS:
        add_li = [('[PAD]', 0)for _ in range(MAX_TOKENS - len(li))]
        li = li + add_li
        
    topics_T.append(li)

In [44]:
# Assign each document with its related topics (train)
# Assign topic distribution over words
# The size is K x V where K is topics related to the document and V is vocab size
# There are 2 methods: with scoring and without scoring
# Scoring: the contribution score of topic to the doc will be multiplied to the topic distribution

def save_topic_dist(corpus_name, files, topic_distr, topics_T):
    is_scoring = False
    path = "../bert_data/id-p2"
    
    data_index = 0
    for file in files:
        filename = file.split("/")[-1]
    
        # Loop only for train files
        if "bert.pt" in file and corpus_name in file:
            print(f"Processing {filename}...")
            bert_data = torch.load(file)  # inside each files contains 2000 data
    
            # Loop through each file
            for i in range(len(bert_data)):
                d = topic_distr[i][:10]
                bert_data[i]['topic_dist'] = []
                distribution_over_words = []
    
                # We make it smaller like this: K x MAX_TOKENS
                # [{token: token_score, token: token_score, ..., MAX_TOKENS},
                # {token: token_score, token: token_score, ..., MAX_TOKENS},
                # {token: token_score, token: token_score, ..., MAX_TOKENS}]
                if len(d) > 0:
                    # Assign topic distribution over words 
                    for item in d:
                        topic_id = item[0]
                        topic_score = item[1]                    
                        if is_scoring:
                            topics_T_scored = []
                            for token_tuple in topics_T[topic_id]:
                                new_tuple = (token_tuple[0], token_tuple[1] * topic_score) # multiply by the contribution score of the topic
                                topics_T_scored.append(new_tuple)
                            distribution_over_words.append(topics_T_scored) 
                        else:
                            distribution_over_words.append(topics_T[topic_id])
                
                # Set the dimension to be equal across documents
                empty_topic = [('[PAD]', 0) for _ in range(MAX_TOKENS)]
                distribution_over_words = distribution_over_words + [empty_topic] * (10 - len(distribution_over_words))
                
                bert_data[i]['topic_dist'] = distribution_over_words
                
                data_index = data_index + 1
            
            torch.save(bert_data, f"./data/lda_topics/{filename}")

In [56]:
len(val_topic_distr)

1

In [55]:
for i in range(len(val_topic_distr)):
    if len(val_topic_distr[i]) == 10:
        print(len(val_topic_distr[i]))

10


In [58]:
save_topic_dist("train", train_files, train_topic_distr, topics_T)

Processing xlsum.train.0.bert.pt...
Processing xlsum.train.1.bert.pt...
Processing xlsum.train.10.bert.pt...
Processing xlsum.train.11.bert.pt...
Processing xlsum.train.12.bert.pt...
Processing xlsum.train.13.bert.pt...
Processing xlsum.train.14.bert.pt...
Processing xlsum.train.15.bert.pt...
Processing xlsum.train.16.bert.pt...
Processing xlsum.train.17.bert.pt...
Processing xlsum.train.18.bert.pt...
Processing xlsum.train.19.bert.pt...
Processing xlsum.train.2.bert.pt...
Processing xlsum.train.3.bert.pt...
Processing xlsum.train.4.bert.pt...
Processing xlsum.train.5.bert.pt...
Processing xlsum.train.6.bert.pt...
Processing xlsum.train.7.bert.pt...
Processing xlsum.train.8.bert.pt...
Processing xlsum.train.9.bert.pt...


In [59]:
save_topic_dist("val", val_files, val_topic_distr, topics_T)

Processing xlsum.valid.0.bert.pt...
Processing xlsum.valid.1.bert.pt...
Processing xlsum.valid.2.bert.pt...


In [60]:
save_topic_dist("test", test_files, test_topic_distr, topics_T)

Processing xlsum.test.0.bert.pt...
Processing xlsum.test.1.bert.pt...
Processing xlsum.test.2.bert.pt...


In [61]:
loaded_data = torch.load("./data/lda_topics/xlsum.test.0.bert.pt")

In [62]:
loaded_data[0]['topic_dist']

[[('foto', 0.028540755),
  ('media', 0.023320349),
  ('sosial', 0.017882474),
  ('twitter', 0.017882403),
  ('akun', 0.016742053),
  ('orang', 0.011105119),
  ('video', 0.010841668),
  ('pengguna', 0.0107230395),
  ('facebook', 0.010492231),
  ('situs', 0.010262921),
  ('pesan', 0.008995027),
  ('tersebut', 0.008917925),
  ('mengatakan', 0.008246847),
  ('kata', 0.006590498),
  ('instagram', 0.0059782504),
  ('yang', 0.005276245),
  ('tidak', 0.005211636),
  ('internet', 0.005137724),
  ('gambar', 0.005088496),
  ('mengunggah', 0.004758432),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('[PAD]', 0),
  ('

In [76]:
import json
for file in os.listdir("../bert_data/topic_id-p2_lda/"):
    if "bert.pt" not in file:
        continue
    filename = "../bert_data/topic_id-p2_lda/" + file
    print(filename)
    datasets = torch.load(filename)
    for i in range(len(datasets)):
        datasets[i]['topic_dist'] = str(datasets[i]['topic_dist'])
        
    filename = filename.replace('bert.pt', 'json')
    with open(filename, 'w') as f:
        f.write(json.dumps(datasets))

../bert_data/topic_id-p2_lda/xlsum.train.19.bert.pt
../bert_data/topic_id-p2_lda/xlsum.test.0.bert.pt
../bert_data/topic_id-p2_lda/xlsum.valid.0.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.12.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.2.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.11.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.15.bert.pt
../bert_data/topic_id-p2_lda/xlsum.valid.2.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.8.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.13.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.17.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.3.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.16.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.1.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.5.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.7.bert.pt
../bert_data/topic_id-p2_lda/xlsum.test.1.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.14.bert.pt
../bert_data/topic_id-p2_lda/xlsum.train.4.bert.pt
../bert_data/topic_id-p2_